# Q1 — o tamanho da diferença: renda e escolaridade dos pais

Objetivo: investigar a diferença nas notas da prova de Ciências da Natureza (CN) do ENEM entre as redes de ensino, as faixas de renda per capita e os níveis de escolaridade dos pais. Essa diferença muda ao longo do tempo?

- **Por renda e escolaridade dos pais:** só de 2020 a 2023 (`fct_individual`), porque em 2024 e 2025 a nota não se liga mais ao questionário socioeconômico, como explicado na D002. É o objeto deste notebook.

- **Por rede:** de 2020 a 2025 (`agg_rede_ano`), com atenção à quebra de série entre 2023 e 2024, também explicada na D002. Essa parte já foi analisada em `01_q1_tamanho_da_diferenca.ipynb`.

## Configurações iniciais

In [ ]:
from src.db import conectar
from src import graficos as g

con = conectar(read_only=True)
g.aplicar_tema()

## Considerações iniciais

A partir de 2024, os microdados do ENEM passaram a ser divididos em duas tabelas: `PARTICIPANTES` e `RESULTADOS`. A primeira traz os dados de perfil e do questionário socioeconômico. A segunda traz os dados de escola, presença, notas, respostas e gabaritos.

O próprio dicionário de 2024 afirma que não é possível relacionar as duas tabelas, porque elas não têm chave em comum. Portanto, não é possível relacionar a nota de um participante com o seu perfil socioeconômico.

Por isso, toda a análise deste notebook se limita ao período de **2020 a 2023**.

Para mais detalhes, consulte a D002 (`../docs/decisoes.md`).

## Análise da diferença por renda

Vamos selecionar as colunas de interesse da `fct_individual`:

In [ ]:
renda = con.sql("""
    SELECT ano, faixa_renda_per_capita,
        COUNT(*) AS qtd_com_nota,
        AVG(nota_cn) AS media,
        MEDIAN(nota_cn) AS mediana,
        STDDEV_SAMP(nota_cn) AS desvio_padrao
    FROM fct_individual
    WHERE tem_nota_cn AND tem_questionario
    GROUP BY ALL
    ORDER BY ano, faixa_renda_per_capita
""").df()
print(renda.round(1).to_string())

A coluna `faixa_renda_per_capita` foi escolhida como referência para as análises de renda. Ela é medida em salários mínimos (SM) e dividida em quatro faixas (até 1/2 SM, de 1/2 a 1 SM, de 1 a 2 SM e acima de 2 SM). Veja os detalhes na D006 (`../docs/decisoes.md`).

Para montar o gráfico de comparação, vamos usar:

In [ ]:
ordem_renda = ["ate_meio_sm", "meio_a_1_sm", "1_a_2_sm", "acima_2_sm"]
rotulos_renda = {"ate_meio_sm": "Até 1/2 SM",
                "meio_a_1_sm": "De 1/2 a 1 SM",
                "1_a_2_sm": "De 1 a 2 SM",
                "acima_2_sm": "Acima de 2 SM"}

In [ ]:
fig, ax = g.serie_temporal(
    renda, x="ano", y="media", grupo="faixa_renda_per_capita",
    ordem=ordem_renda, rotulos=rotulos_renda, sequencial=True,
    legenda=False,
    titulo="Média em Ciências da Natureza por renda per capita — MG, 2020–2023",
    subtitulo="Concluintes com nota na prova e questionário respondido",
    rotulo_y="Nota média (pontos)",
    nota="Fonte: microdados do ENEM (INEP). SM: salário mínimo do ano. "
        "Renda per capita estimada pelo ponto médio da faixa (D006).",
    figsize=(8, 5),
)
g.salvar(fig, "q1_media_cn_por_renda", dpi=300)

O gráfico mostra uma diferença clara e regular entre as faixas de renda. No entanto, é preciso considerar que, provavelmente, a maioria dos alunos de renda alta estuda nas redes privada e federal, e os alunos de renda mais baixa estudam, com frequência, na rede estadual/municipal. Então, parte da diferença por renda pode ser a diferença por rede, e vice-versa.

### A distância entre as faixas extremas, ano a ano

Vamos analisar, ano a ano, como a média em Ciências da Natureza varia entre as faixas. Na tabela a seguir, a coluna `extremos` mostra a diferença entre a média da faixa de renda per capita mais alta e a da mais baixa, e as colunas `passo_1` a `passo_3` mostram a diferença entre cada faixa e a faixa imediatamente abaixo.

In [ ]:
piv = renda.pivot(index="ano", columns="faixa_renda_per_capita", values="media")[ordem_renda]
gap_renda = piv.assign(
    extremos=piv["acima_2_sm"] - piv["ate_meio_sm"],
    passo_1=piv["meio_a_1_sm"] - piv["ate_meio_sm"],
    passo_2=piv["1_a_2_sm"] - piv["meio_a_1_sm"],
    passo_3=piv["acima_2_sm"] - piv["1_a_2_sm"],
).round(1)
print(gap_renda.to_string())

Também podemos medir a diferença entre as faixas extremas numa escala comum, usando o **d de Cohen**:

In [ ]:
ext = renda[renda["faixa_renda_per_capita"].isin(["ate_meio_sm", "acima_2_sm"])]
ext = ext.pivot(index="ano", columns="faixa_renda_per_capita",
                values=["media", "desvio_padrao", "qtd_com_nota"])
n1, n2 = ext[("qtd_com_nota", "ate_meio_sm")], ext[("qtd_com_nota", "acima_2_sm")]
s1, s2 = ext[("desvio_padrao", "ate_meio_sm")], ext[("desvio_padrao", "acima_2_sm")]
dp_comb = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
d = (ext[("media", "acima_2_sm")] - ext[("media", "ate_meio_sm")]) / dp_comb
print(d.round(2).to_string())

Os valores do *d* de Cohen entre as faixas extremas, ano a ano, estão na tabela:

| Ano | *d* de Cohen |
|---|---|
| 2020 | 1,47 |
| 2021 | 1,39 |
| 2022 | 1,24 |
| 2023 | 1,38 |

O *d* de Cohen entre as faixas extremas fica entre 1,24 e 1,47, bem acima de 0,8, o valor de referência para um efeito "grande". Um *d* em torno de 1,4 indica que um aluno típico da faixa acima de 2 SM tem nota maior que a de cerca de 9 em cada 10 alunos da faixa até 1/2 SM. Em outras palavras, cerca de 92% dos alunos da faixa de renda mais baixa têm nota abaixo da média da faixa mais alta.

Os passos entre faixas vizinhas ficam entre 25 e 38 pontos, em torno de 30 na maioria dos anos. A média sobe de forma regular de uma faixa para a outra, sem um salto concentrado num ponto. Como os limites das faixas dobram de uma faixa para a seguinte (0,5, 1 e 2 SM, pela definição da D006), e isso corresponde a passos parecidos na nota, pode ser que a nota cresça com o logaritmo da renda. Essa hipótese será explorada na Q3.

O ano de 2022 tem o menor gap entre os extremos (80,2 pontos) e o menor *d* (1,24). Se a prova de 2022 tivesse apenas encolhido a escala das notas, o gap em pontos cairia, mas o *d* ficaria parecido, porque o desvio-padrão encolheria junto. Como o *d* também cai, a diferença entre os grupos em 2022 diminuiu mais do que a dispersão geral. Esse efeito também pode ser observado no gráfico.

O *d* de Cohen também permite comparar a renda com a rede numa escala comum:

In [ ]:
r = con.sql("""
    SELECT ano, rede, media_nota_cn AS media,
        desvio_padrao_nota_cn AS dp, qtd_com_nota_cn AS n
    FROM agg_rede_ano
    WHERE periodo = '2020-2023' AND rede IN ('privada', 'estadual_municipal')
""").df().pivot(index="ano", columns="rede")
n1, n2 = r[("n", "estadual_municipal")], r[("n", "privada")]
s1, s2 = r[("dp", "estadual_municipal")], r[("dp", "privada")]
dp_comb = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
d_rede = (r[("media", "privada")] - r[("media", "estadual_municipal")]) / dp_comb
print(d_rede.round(2).to_string())

Com isso, podemos comparar a diferença por renda com a diferença por rede:

| Ano | *d* por renda (extremos) | *d* por rede (privada × estadual/municipal) |
|----|------------------------|----------------------------|
| 2020 | 1,47 | 1,18 |
| 2021 | 1,39 | 1,16 |
| 2022 | 1,24 | 1,01 |
| 2023 | 1,38 | 1,09 |

Tanto a diferença por renda quanto a diferença por rede são grandes. A diferença por renda é maior que a diferença por rede em todos os anos, por uma margem de 0,2 a 0,3. E, mais uma vez, 2022 tem o menor valor nas duas colunas. Apesar de esse resultado chamar a atenção, ele tem uma limitação.

O *d* da rede vem da `agg_rede_ano` (concluintes com escola mineira informada). Já o *d* da renda vem da `fct_individual` (todos os concluintes com questionário, com ou sem escola informada). Para comparar na mesma população, precisamos recalcular o *d* da rede na `fct_individual`, filtrando quem tem escola:

In [ ]:
r2 = con.sql("""
    SELECT ano, rede,
        AVG(nota_cn) AS media, STDDEV_SAMP(nota_cn) AS dp, COUNT(*) AS n
    FROM fct_individual
    WHERE tem_nota_cn AND tem_escola
    AND rede IN ('privada', 'estadual_municipal')
    GROUP BY ALL
""").df().pivot(index="ano", columns="rede")
n1, n2 = r2[("n", "estadual_municipal")], r2[("n", "privada")]
s1, s2 = r2[("dp", "estadual_municipal")], r2[("dp", "privada")]
dp_comb = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
d_rede_ind = (r2[("media", "privada")] - r2[("media", "estadual_municipal")]) / dp_comb
print(d_rede_ind.round(2).to_string())

O *d* é o mesmo, pelo menos até a segunda casa decimal. Isso confirma que a `fct_individual` com escola preenchida e a `fct_agregada` são a mesma população em 2020–2023, como a D002 já indicava. Mas isso resolve só metade do problema, porque as populações dos dois *d* continuam diferentes:

- o *d* da rede: concluintes com escola informada;
- o *d* da renda: todos os concluintes com questionário, com ou sem escola informada.

Para encerrar a questão, vamos calcular o *d* da renda só entre quem tem escola informada, a mesma população do *d* da rede:

In [ ]:
import pandas as pd

comparacao = pd.DataFrame({
    "d_renda_extremos": d,
    "d_rede_agg": d_rede,
    "d_rede_individual": d_rede_ind,
}).round(2)

In [ ]:
renda_esc = con.sql("""
    SELECT ano, faixa_renda_per_capita,
        AVG(nota_cn) AS media, STDDEV_SAMP(nota_cn) AS dp, COUNT(*) AS n
    FROM fct_individual
    WHERE tem_nota_cn AND tem_questionario AND tem_escola
    AND faixa_renda_per_capita IN ('ate_meio_sm', 'acima_2_sm')
    GROUP BY ALL
""").df().pivot(index="ano", columns="faixa_renda_per_capita")

n1, n2 = renda_esc[("n", "ate_meio_sm")], renda_esc[("n", "acima_2_sm")]
s1, s2 = renda_esc[("dp", "ate_meio_sm")], renda_esc[("dp", "acima_2_sm")]
dp_comb = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
d_renda_esc = (renda_esc[("media", "acima_2_sm")] - renda_esc[("media", "ate_meio_sm")]) / dp_comb

comparacao["d_renda_com_escola"] = d_renda_esc.round(2)
print(comparacao.to_string())

| Ano | *d* da renda (com escola) | *d* da rede | Diferença |
|------|-----------------|-------|-----------|
|2020|1,35|1,18|+0,17|
|2021|1,29|1,16|+0,13|
|2022|1,16|1,01|+0,15|
|2023|1,26|1,09|+0,17|

Na mesma população, a diferença por renda continua maior que a diferença por rede em todos os anos, mas por uma margem bem menor do que antes (de 0,13 a 0,17, contra 0,2 a 0,3).

O *d* da renda cai cerca de 0,1 quando se excluem os concluintes sem escola informada. Ou seja, incluir esse grupo aumenta a diferença entre os extremos de renda. Isso indica que os concluintes sem escola informada não são um grupo neutro: eles parecem ampliar a desigualdade observada. É mais uma evidência para a ressalva da D002 de que esse grupo provavelmente não é aleatório.

Há ainda duas limitações. O *d* da renda compara os extremos de quatro faixas (deixando as duas do meio de fora), enquanto o *d* da rede compara duas redes inteiras, o que tende a favorecer a renda. Além disso, renda e rede se sobrepõem, e a separação dos seus efeitos fica para a Q3.

## Análise por escolaridade dos pais

Partimos agora para a análise pela escolaridade dos pais. De acordo com a D006, a análise usa a maior escolaridade entre o pai e a mãe. Ela tem quatro níveis ordenados e uma categoria à parte:

| Categoria | Significado |
|----------|-------------|
| `fund_incompleto` | fundamental incompleto |
| `fund_completo` | fundamental completo |
| `medio_completo` | médio completo |
| `superior_completo` | superior completo |
| `nao_sei` | o aluno não sabe a escolaridade de nenhum dos dois (fora da escala, D005) |

In [ ]:
esc = con.sql("""
    SELECT ano, escolaridade_max_pais_cat AS escolaridade,
        COUNT(*) AS qtd_com_nota,
        AVG(nota_cn) AS media,
        MEDIAN(nota_cn) AS mediana,
        STDDEV_SAMP(nota_cn) AS desvio_padrao
    FROM fct_individual
    WHERE tem_nota_cn AND tem_questionario
    GROUP BY ALL
    ORDER BY ano, escolaridade
""").df()
print(esc.round(1).to_string())

In [ ]:
ordem_esc = ["fund_incompleto", "fund_completo", "medio_completo",
            "superior_completo", "nao_sei"]
rotulos_esc = {"fund_incompleto": "Fundamental incompleto",
            "fund_completo": "Fundamental completo",
            "medio_completo": "Médio completo",
            "superior_completo": "Superior completo",
            "nao_sei": "Não sabe"}

In [ ]:
fig, ax = g.serie_temporal(
    esc, x="ano", y="media", grupo="escolaridade",
    ordem=ordem_esc, rotulos=rotulos_esc, sequencial=True,
    legenda=True,
    titulo="Média em Ciências da Natureza pela escolaridade dos pais — MG, 2020–2023",
    subtitulo="Maior escolaridade entre pai e mãe; concluintes com nota e questionário respondido",
    rotulo_y="Nota média (pontos)",
    nota="Fonte: microdados do ENEM (INEP). \"Não sabe\": o aluno não sabe a "
        "escolaridade de nenhum dos pais (D005).",
    figsize=(8, 5),
)
g.salvar(fig, "q1_media_cn_por_escolaridade_pais")

O gráfico mostra que, quanto maior a escolaridade dos pais, maior a nota média do participante. As médias dos participantes cuja maior escolaridade dos pais é o fundamental incompleto ou o fundamental completo ficam próximas entre si de 2020 a 2023. A média dos participantes que marcaram a opção "não sei" fica acima da média dos filhos de pais que não concluíram o ensino fundamental, exceto em 2022.

Vamos observar agora os passos entre níveis vizinhos e a distância entre os extremos:

In [ ]:
niveis = ["fund_incompleto", "fund_completo", "medio_completo", "superior_completo"]

piv_esc = esc.pivot(index="ano", columns="escolaridade", values="media")
gap_esc = piv_esc[niveis].assign(
    extremos=piv_esc["superior_completo"] - piv_esc["fund_incompleto"],
    passo_1=piv_esc["fund_completo"] - piv_esc["fund_incompleto"],
    passo_2=piv_esc["medio_completo"] - piv_esc["fund_completo"],
    passo_3=piv_esc["superior_completo"] - piv_esc["medio_completo"],
    nao_sei_menos_fund_inc=piv_esc["nao_sei"] - piv_esc["fund_incompleto"],
).round(1)
print(gap_esc.to_string())

A categoria `nao_sei` fica fora dos passos, mas ganha uma coluna própria, que a compara com a categoria `fund_incompleto`.

A distância entre os extremos varia de 58,4 a 73,8 pontos. Em geral, 2020 é o ano com os maiores passos entre os níveis de escolaridade. A diferença entre `nao_sei` e `fund_incompleto` é muito pequena, de −1,4 (em 2022, quando `fund_incompleto` ultrapassa `nao_sei`) a 3,6 pontos. Isso sugere que os participantes que não sabem a escolaridade dos pais têm desempenho parecido com o dos filhos de pais com a menor escolaridade.

Diferentemente do que acontece com a renda, os passos da escolaridade não são regulares. A maior parte da diferença se concentra na passagem para o ensino superior: o último passo (de 36 a 45 pontos) é de 2,6 a 2,8 vezes maior que o segundo e de 3,6 a 5,5 vezes maior que o primeiro.

Assim como foi feito para a renda, vamos calcular o *d* de Cohen, tanto para todos os concluintes com questionário quanto para a população com escola informada:

In [ ]:
def d_cohen(df, grupo_col, baixo, alto):
    """d de Cohen entre dois grupos, por ano.
    df precisa ter as colunas: ano, <grupo_col>, media, dp, n."""
    p = df[df[grupo_col].isin([baixo, alto])].pivot(index="ano", columns=grupo_col)
    n1, n2 = p[("n", baixo)], p[("n", alto)]
    s1, s2 = p[("dp", baixo)], p[("dp", alto)]
    dp_comb = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
    return ((p[("media", alto)] - p[("media", baixo)]) / dp_comb).round(2)

In [ ]:
consulta_esc = """
    SELECT ano, escolaridade_max_pais_cat AS escolaridade,
        AVG(nota_cn) AS media, STDDEV_SAMP(nota_cn) AS dp, COUNT(*) AS n
    FROM fct_individual
    WHERE tem_nota_cn AND tem_questionario {filtro}
    GROUP BY ALL
"""
esc_todos = con.sql(consulta_esc.format(filtro="")).df()
esc_escola = con.sql(consulta_esc.format(filtro="AND tem_escola")).df()

comparacao["d_escolaridade_extremos"] = d_cohen(
    esc_todos, "escolaridade", "fund_incompleto", "superior_completo")
comparacao["d_escolaridade_com_escola"] = d_cohen(
    esc_escola, "escolaridade", "fund_incompleto", "superior_completo")
print(comparacao.to_string())

Na população com escola informada, a mesma usada para a rede:

| Ano | *d* rede | *d* renda | *d* escolaridade |
|------|---------|----------|-----------------|
| 2020 | 1,18 | 1,35 | 1,01 |
| 2021 | 1,16 | 1,29 | 0,88 |
| 2022 | 1,01 | 1,16 | 0,85 |
| 2023 | 1,09 | 1,26 | 0,91 |

Os *d* dos três fatores mostram uma ordem que se repete em todos os anos: a renda per capita tem sempre o maior *d*, seguida pela rede de ensino e, depois, pela escolaridade dos pais. Os três ficam acima de 0,8, a referência para um efeito "grande".

Vale ressaltar, porém, que o *d* de Cohen depende de quão "extremos" são os grupos comparados. Na renda, a faixa de cima (acima de 2 SM) reúne cerca de 10% dos participantes. Na escolaridade, o nível de cima (superior completo) reúne cerca de 36%, um grupo mais amplo e, portanto, menos extremo. Isso tende a dar um *d* menor para a escolaridade, sem significar que ela "importe menos".

Além disso, os padrões de ano se repetem: o menor *d* em 2022 e a queda das médias de 2020 para 2021 em todos os níveis. Com rede, renda e escolaridade mostrando o mesmo padrão, fica mais forte a leitura de que esses efeitos vêm do ano (a prova ou o contexto), e não dos grupos.

### Distribuição das notas por renda e por escolaridade

As médias resumem cada grupo num número. Para ver a forma das diferenças, vamos olhar as distribuições inteiras em 2023, o ano mais recente da análise individual.

In [ ]:
notas_ind = con.sql("""
    SELECT faixa_renda_per_capita, escolaridade_max_pais_cat AS escolaridade, nota_cn
    FROM fct_individual
    WHERE ano = 2023 AND tem_nota_cn AND tem_questionario
""").df()

In [ ]:
fig, ax = g.distribuicoes(
    notas_ind,
    coluna_valor="nota_cn",
    coluna_grupo="faixa_renda_per_capita",
    ordem=ordem_renda,
    rotulos=rotulos_renda,
    sequencial=True,
    mostrar_n=True,
    intervalo=(290, 850),
    preencher=False,
    titulo="Distribuição das notas em Ciências da Natureza por renda per capita — MG, 2023",
    subtitulo="Concluintes com nota na prova e questionário respondido",
    rotulo_x="Nota em Ciências da Natureza (pontos)",
    rotulo_y="Densidade",
    nota="Fonte: microdados do ENEM (INEP). Curvas normalizadas: cada faixa tem área 1.",
)
g.salvar(fig, "q1_distribuicao_cn_por_renda_2023")

As curvas se deslocam para a direita de forma regular, acompanhando as médias de 2023 (483,9 em "até 1/2 SM", 514,5 em "de 1/2 a 1 SM", 544,8 em "de 1 a 2 SM" e 579,8 em "acima de 2 SM"). É a versão visual dos passos regulares de cerca de 30 pontos.

As curvas das faixas mais altas são mais largas. A faixa "acima de 2 SM" se espalha mais, sobretudo para a direita, e só as faixas de renda mais alta chegam com alguma densidade a 700–800 pontos, o que é coerente com os desvios-padrão (cerca de 82 pontos, contra 67 na faixa mais baixa).

A sobreposição entre as faixas é grande. Mesmo entre os extremos, há uma faixa larga de notas, mais ou menos de 480 a 620 pontos, em que as duas curvas têm bastante densidade. Ou seja: muitos alunos da faixa mais pobre têm notas iguais ou maiores que as de muitos alunos da faixa mais rica. Podemos medir essa sobreposição:

In [ ]:
baixo = notas_ind.loc[notas_ind["faixa_renda_per_capita"] == "ate_meio_sm", "nota_cn"]
alto = notas_ind.loc[notas_ind["faixa_renda_per_capita"] == "acima_2_sm", "nota_cn"]
print(f"{(baixo > alto.median()).mean():.1%} dos alunos até 1/2 SM ficam acima da mediana de quem tem mais de 2 SM")
print(f"{(alto < baixo.median()).mean():.1%} dos alunos acima de 2 SM ficam abaixo da mediana de quem tem até 1/2 SM")

Em 2023, 6,8% dos alunos da faixa até 1/2 SM ficam acima da mediana da faixa acima de 2 SM, e 12,0% dos alunos acima de 2 SM ficam abaixo da mediana da faixa até 1/2 SM. Se as distribuições não se sobrepusessem, os dois valores seriam 0%; se fossem idênticas, seriam 50%. A sobreposição existe, mas é limitada: o grupo descreve uma tendência forte, sem determinar a nota de cada aluno.

O pico perto de 340 pontos aparece novamente, como no gráfico de distribuição por rede (`01_q1_tamanho_da_diferenca.ipynb`), e diminui conforme a renda aumenta: é mais visível em "até 1/2 SM" e quase some em "acima de 2 SM".

In [ ]:
fig, ax = g.distribuicoes(
    notas_ind,
    coluna_valor="nota_cn",
    coluna_grupo="escolaridade",
    ordem=ordem_esc,
    rotulos=rotulos_esc,
    sequencial=True,
    mostrar_n=True,
    intervalo=(290, 850),
    preencher=False,
    titulo="Distribuição das notas em Ciências da Natureza pela escolaridade dos pais — MG, 2023",
    subtitulo="Maior escolaridade entre pai e mãe; concluintes com nota e questionário respondido",
    rotulo_x="Nota em Ciências da Natureza (pontos)",
    rotulo_y="Densidade",
    nota="Fonte: microdados do ENEM (INEP). Curvas normalizadas: cada grupo tem área 1. "
        "\"Não sabe\": o aluno não sabe a escolaridade de nenhum dos pais (D005).",
)
g.salvar(fig, "q1_distribuicao_cn_por_escolaridade_2023")

No gráfico da escolaridade, as três primeiras curvas praticamente se sobrepõem. Fundamental incompleto, fundamental completo e médio completo têm picos próximos (entre 460 e 500 pontos), com o médio completo só um pouco à direita. Essa observação é coerente com os passos pequenos vistos na tabela de passos.

A curva do superior completo se destaca: ela tem o pico mais à direita (perto de 560 pontos), é a mais larga e tem a cauda mais longa para as notas altas. É a imagem do salto de 36 a 45 pontos no último passo.

O "Não sabe" acompanha o fundamental incompleto quase por inteiro, confirmando o que as médias mostravam.

O pico perto de 340 pontos aparece nos três grupos de baixo e no "Não sabe", e quase não aparece no superior completo.

Comparando as distribuições da renda e da escolaridade: na renda, cada degrau conta; na escolaridade dos pais, o que mais separa as notas é ter ou não um pai ou uma mãe com ensino superior.

Também podemos medir a sobreposição entre os extremos da escolaridade:

In [ ]:
baixo_esc = notas_ind.loc[notas_ind["escolaridade"] == "fund_incompleto", "nota_cn"]
alto_esc = notas_ind.loc[notas_ind["escolaridade"] == "superior_completo", "nota_cn"]
print(f"{(baixo_esc > alto_esc.median()).mean():.1%} dos alunos com pais de fundamental incompleto "
    f"ficam acima da mediana de quem tem pais com superior completo")
print(f"{(alto_esc < baixo_esc.median()).mean():.1%} dos alunos com pais de superior completo "
    f"ficam abaixo da mediana de quem tem pais com fundamental incompleto")

Em 2023, 15,1% dos filhos de pais com fundamental incompleto ficam acima da mediana dos filhos de pais com superior completo, e 20,5% dos filhos de pais com superior completo ficam abaixo da mediana do outro grupo. A sobreposição é bem maior que a da renda (6,8% e 12,0%), o que é coerente com o *d* menor da escolaridade: as duas medidas contam a mesma história por caminhos diferentes.

## Conclusões da análise por renda e escolaridade dos pais

### O que os dados mostram

- **A diferença por renda é grande e gradual.** A nota média sobe cerca de 30 pontos a cada faixa de renda per capita, e a distância entre as faixas extremas vai de 80 a 104 pontos. O *d* de Cohen entre os extremos fica entre 1,24 e 1,47 (de 1,16 a 1,35 entre os concluintes com escola informada).
- **A diferença por escolaridade dos pais se concentra no ensino superior.** Do fundamental incompleto ao médio completo, os passos são pequenos (de 8 a 17 pontos); a passagem para o superior completo acrescenta de 36 a 45 pontos. O *d* entre os extremos fica entre 0,82 e 0,95 (de 0,85 a 1,01 com escola informada). Os alunos que não sabem a escolaridade dos pais têm desempenho parecido com o dos filhos de pais com fundamental incompleto.
- **Na mesma população, a ordem se repete em todos os anos:** renda > rede > escolaridade dos pais, com os três *d* acima de 0,8.
- **As distribuições se sobrepõem.** Em 2023, 6,8% dos alunos da faixa de renda mais baixa ficam acima da mediana da faixa mais alta, e 15,1% dos filhos de pais com fundamental incompleto ficam acima da mediana dos filhos de pais com superior completo. Os grupos descrevem tendências fortes, mas não determinam a nota de cada aluno.
- **Os padrões de ano se repetem nos três fatores:** a menor diferença em 2022 e a queda das médias de 2020 para 2021 em todos os grupos, o que sugere efeitos do ano (a prova ou o contexto), e não dos grupos.

### O que exige cuidado na leitura

- **Associação não é causa.** As diferenças descrevem os grupos, e não o efeito da renda ou da escolaridade sobre a nota.
- **Os três fatores se sobrepõem.** Renda, escolaridade dos pais e rede de ensino estão ligadas entre si e descrevem, em boa parte, o mesmo gradiente social. Separar a contribuição de cada um é o objetivo da Q3.
- **O *d* depende de como os grupos são definidos.** Os extremos da renda (cerca de 10% dos alunos na faixa de cima) são mais extremos que os da escolaridade (cerca de 36% no superior completo), e a comparação com a rede usa duas redes inteiras.
- **A renda per capita é uma estimativa**, calculada pelo ponto médio da faixa e pelo número de moradores (D006).
- **A população importa.** Os concluintes sem escola informada ampliam a diferença por renda: o *d* cai cerca de 0,1 quando eles são excluídos.

### O que fica em aberto

- **O pico perto de 340 pontos**, mais forte nos grupos de menor renda e escolaridade. Uma hipótese é o piso da escala da TRI, em que se acumulam as notas de quem acerta poucas questões ou responde no chute. Ela pode ser testada com os vetores de respostas.
- **A relação logarítmica entre nota e renda**, sugerida pelos passos regulares em faixas que dobram de largura, a ser testada na Q3.
- **O que torna 2022 diferente**, já que as diferenças entre grupos encolhem nesse ano em todos os fatores.